In [ ]:
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point

# Cargamos archivos

In [ ]:
# Configuración de visualización
sns.set_style("whitegrid")

# Cargamos raíz
root = r"C:\Users\fbetancourt\Documents\GitHub\Tesis\Datasets\gtfs_amg_20240312\Datos"

# Cargar los archivos CSV
files = {
    "trips": r"\trips.csv",
    "stop_times": r"\stop_times.csv",
    "frequencies": r"\frequencies.csv",
    "shapes": r"\shapes.csv",
    "agency": r"\agency.csv",
    "fare_attributes": r"\fare_attributes.csv",
    "calendar_dates": r"\calendar_dates.csv",
    "routes": r"\routes.csv",
    "calendar": r"\calendar.csv",
    "stops": r"\stops.csv"
}

In [ ]:
# Función para análisis general
def analyze_file(file_name, dataset):
    print(f"\n--- Análisis Exploratorio de {file_name} ---")
    print(dataset.info())
    print(dataset.head())
    print(dataset.describe())
    print("Valores nulos por columna:")
    print(dataset.isnull().sum())

# Evaluamos un resumen de su contenido

In [ ]:
# Cargar y analizar cada archivo con mejor presentación
dataframes = {}  # Guardar DataFrames para análisis posteriores

for name, file in files.items():
    df = pd.read_csv(root + file)
    dataframes[name] = df
    print(f"\n📂 Archivo cargado: {name.upper()} (Filas: {df.shape[0]}, Columnas: {df.shape[1]})")

    # Mostrar solo el resumen con DataFrame en lugar de texto
    display(df.head())  # Muestra las primeras 5 filas
    display(df.describe())  # Muestra estadísticas de columnas numéricas
    display(df.info())  # Muestra información estructural del DataFrame

    # Valores nulos en formato tabular para mejor legibilidad
    null_values = df.isnull().sum()
    null_values = null_values[null_values > 0]  # Filtrar solo columnas con valores nulos
    if not null_values.empty:
        print("\n🔍 Valores nulos por columna:")
        display(pd.DataFrame(null_values, columns=["Valores nulos"]))


# Trips

In [ ]:
df_trips = dataframes["trips"]
df_trips

Existen rutas complementarias, alimentadoras y troncales

In [ ]:
inicial =

# Shapes

In [ ]:
# Cargar datos de shapes
df_shapes = dataframes["shapes"].copy()  # Crear copia para evitar modificar el original

# Crear geometría con latitud y longitud
df_shapes["geometry"] = df_shapes.apply(lambda row: Point(row["shape_pt_lon"], row["shape_pt_lat"]), axis=1)

# Definir sistema de referencia WGS 84 (EPSG:4326)
gdf_shapes = gpd.GeoDataFrame(df_shapes, geometry="geometry", crs="EPSG:4326")

# Definir límites de la Zona Metropolitana de Guadalajara (ZMG)
ZMG_BOUNDING_BOX = {
    "min_lon": -103.5, "max_lon": -103.2,
    "min_lat": 20.5, "max_lat": 20.8
}

# Filtrar solo los puntos dentro de la ZMG
gdf_shapes_zmg = gdf_shapes[
    (gdf_shapes["shape_pt_lon"] >= ZMG_BOUNDING_BOX["min_lon"]) &
    (gdf_shapes["shape_pt_lon"] <= ZMG_BOUNDING_BOX["max_lon"]) &
    (gdf_shapes["shape_pt_lat"] >= ZMG_BOUNDING_BOX["min_lat"]) &
    (gdf_shapes["shape_pt_lat"] <= ZMG_BOUNDING_BOX["max_lat"])
].copy()

# Cargar un mapa base de México desde GeoPandas (usando "naturalearth_lowres")
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

# Filtrar solo México
mexico = world[world["name"] == "Mexico"]

# Graficar el mapa de México como fondo
fig, ax = plt.subplots(figsize=(10, 6))
mexico.plot(ax=ax, color="lightgrey", edgecolor="black")

# Graficar solo la zona de Guadalajara
gdf_shapes_zmg.plot(ax=ax, markersize=1, color="red", alpha=0.5)

# Ajustar los límites de la gráfica para Guadalajara
ax.set_xlim(ZMG_BOUNDING_BOX["min_lon"], ZMG_BOUNDING_BOX["max_lon"])
ax.set_ylim(ZMG_BOUNDING_BOX["min_lat"], ZMG_BOUNDING_BOX["max_lat"])

# Etiquetas y título
plt.title("Distribución geográfica de las rutas en Guadalajara")
plt.xlabel("Longitud")
plt.ylabel("Latitud")

plt.show()


# Stops

In [ ]:
# Cargar datos de paradas
df_stops = dataframes["stops"].copy()  # Copia para evitar modificar el original

# Crear geometría con latitud y longitud
df_stops["geometry"] = df_stops.apply(lambda row: Point(row["stop_lon"], row["stop_lat"]), axis=1)

# Definir sistema de referencia WGS 84 (EPSG:4326)
gdf_stops = gpd.GeoDataFrame(df_stops, geometry="geometry", crs="EPSG:4326")

# Definir límites de la Zona Metropolitana de Guadalajara (ZMG)
ZMG_BOUNDING_BOX = {
    "min_lon": -103.5, "max_lon": -103.2,
    "min_lat": 20.5, "max_lat": 20.8
}

# Filtrar solo las paradas dentro de la ZMG
gdf_stops_zmg = gdf_stops[
    (gdf_stops["stop_lon"] >= ZMG_BOUNDING_BOX["min_lon"]) &
    (gdf_stops["stop_lon"] <= ZMG_BOUNDING_BOX["max_lon"]) &
    (gdf_stops["stop_lat"] >= ZMG_BOUNDING_BOX["min_lat"]) &
    (gdf_stops["stop_lat"] <= ZMG_BOUNDING_BOX["max_lat"])
].copy()

# Cargar un mapa base de México desde GeoPandas (usando "naturalearth_lowres")
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

# Filtrar solo México
mexico = world[world["name"] == "Mexico"]

# Graficar el mapa de México como fondo
fig, ax = plt.subplots(figsize=(10, 6))
mexico.plot(ax=ax, color="lightgrey", edgecolor="black")

# Graficar solo la zona de Guadalajara
gdf_stops_zmg.plot(ax=ax, markersize=3, color="blue", alpha=0.5)

# Ajustar los límites de la gráfica para Guadalajara
ax.set_xlim(ZMG_BOUNDING_BOX["min_lon"], ZMG_BOUNDING_BOX["max_lon"])
ax.set_ylim(ZMG_BOUNDING_BOX["min_lat"], ZMG_BOUNDING_BOX["max_lat"])

# Etiquetas y título
plt.title("Ubicación geográfica de paradas en Guadalajara")
plt.xlabel("Longitud")
plt.ylabel("Latitud")

plt.show()


# Probamos gtfs functions

In [ ]:
from gtfs_functions import Feed

In [ ]:
gtfs_path = r'C:\Users\fbetancourt\Documents\GitHub\Tesis\Datasets\gtfs_amg_20240312\Datos\gtfs_eadible.zip'


feed = Feed(gtfs_path)

In [ ]:
routes = feed.routes
routes

In [ ]:
stops = feed.stops
stops

In [ ]:
stop_times = feed.stop_times
stop_times

In [ ]:
shapes = feed.shapes
shapes

In [ ]:
time_windows = [0, 6, 9, 15.5, 19, 22, 24]

feed = Feed(gtfs_path, time_windows=time_windows)
stop_freq = feed.stops_freq
stop_freq

In [ ]:
line_freq = feed.lines_freq
line_freq

In [ ]:
segments_gdf = feed.segments
segments_gdf

In [ ]:
# Cutoffs to make get hourly values
speeds = feed.avg_speeds
speeds

In [ ]:
segments_freq = feed.segments_freq
segments_freq

In [ ]:
# Stops
from gtfs_functions.gtfs_plots import map_gdf

condition_dir = stop_freq.dir_id == 'Inbound'
condition_window = stop_freq.window == '6:00-9:00'

gdf = stop_freq.loc[(condition_dir & condition_window),:].reset_index()

map_gdf(
  gdf = gdf,
  variable = 'ntrips',
  colors = ["#d13870", "#e895b3" ,'#55d992', '#3ab071', '#0e8955','#066a40'],
  tooltip_var = ['min_per_trip'] ,
  tooltip_labels = ['Frequency: '],
  breaks = [10, 20, 30, 40, 120, 200]
)

In [ ]:
# Line frequencies
from gtfs_functions.gtfs_plots import map_gdf

condition_dir = line_freq.direction_id == 'Inbound'
condition_window = line_freq.window == '6:00-9:00'

gdf = line_freq.loc[(condition_dir & condition_window),:].reset_index()

map_gdf(
  gdf = gdf,
  variable = 'ntrips',
  colors = ["#d13870", "#e895b3" ,'#55d992', '#3ab071', '#0e8955','#066a40'],
  tooltip_var = ['route_name'] ,
  tooltip_labels = ['Route: '],
  breaks = [5, 10, 20, 50]
)

In [ ]:
# Histogram
import plotly.express as px
px.histogram(
    stop_freq.loc[stop_freq.min_per_trip<50],
    x='frequency',
    title='Stop frequencies',
    template='simple_white',
    nbins =20)

In [ ]:
# Heatmap
import plotly.graph_objects as go
dir_0 = speeds.loc[(speeds.dir_id=='Inbound')&(speeds.route_name=='1 CALIFORNIA')].sort_values(by='stop_sequence')
dir_0['hour'] = dir_0.window.apply(lambda x: int(x.split(':')[0]))
dir_0.sort_values(by='hour', ascending=True, inplace=True)

fig = go.Figure(data=go.Heatmap(
                   z=dir_0.speed_kmh,
                   y=dir_0.start_stop_name,
                   x=dir_0.window,
                   hoverongaps = False,
                   colorscale=px.colors.colorbrewer.RdYlBu,
                   reversescale=False
))

fig.update_yaxes(title_text='Stop', autorange='reversed')
fig.update_xaxes(title_text='Hour of day', side='top')
fig.update_layout(showlegend=False, height=600, width=1000,
                 title='Speed heatmap per direction and hour of the day')

fig.show()

In [ ]:
by_hour = speeds.pivot_table('speed_kmh', index = ['window'], aggfunc = ['mean','std'] ).reset_index()
by_hour.columns = ['_'.join(col).strip() for col in by_hour.columns.values]
by_hour['hour'] = by_hour.window_.apply(lambda x: int(x.split(':')[0]))
by_hour.sort_values(by='hour', ascending=True, inplace=True)

# Scatter
fig = px.line(by_hour,
           x='window_',
           y='mean_speed_kmh',
           template='simple_white',
           #error_y = 'std_speed_kmh'
                )

fig.update_yaxes(rangemode='tozero')

fig.show()